## Plan for Data cleaning based of EDA



In [1]:
import pandas as pd
df = pd.read_csv("Titanic-Dataset.csv")

In [16]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [ ]:
## Embarked has only 2 missing values, so we fill them using the mode (most frequent value)
df["Embarked"].value_counts() ## those two missing rows survivied value is 1 means both are survivied and most of the survived from "S" and also it is majority and so missing value filled with most frequenct

Embarked
S    644
C    168
Q     77
Name: count, dtype: int64

In [19]:
df["Embarked"].fillna(df['Embarked'].mode()[0], inplace=True)

In [20]:
df["Embarked"].isnull().sum()

np.int64(0)

In [26]:
## now cabin (77%) are missing values so filling without mean, mode, etc create fake values most, so instead we create a new column 'HasCabin'
df["HasCabin"] = df["Cabin"].apply(lambda x:  0 if pd.isnull(x) else 1 ) # if missing then 0 else 1
#  df["HasCabin"] = df["Cabin"].notnull().astype(int)

In [27]:
df.drop(columns=["Cabin"], inplace=True)
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Embarked         0
HasCabin         0
dtype: int64

In [ ]:
# now age
# Age has 177 missing values (~20%)
# We cannot use a global median because different Pclass groups have different age profiles
# (1st class passengers are older, 3rd class are younger)
# So we fill each missing Age with the median age of its own Pclass group
df["Age"] = df.groupby("Pclass")["Age"].transform(lambda x: x.fillna(x.median()))

In [31]:
df.isnull().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
HasCabin       0
dtype: int64

In [32]:
(df["Fare"] == 0).sum()

np.int64(15)

In [33]:
print(df[df['Fare'] > 0].groupby('Pclass')['Fare'].median())

Pclass
1    61.9792
2    15.0229
3     8.0500
Name: Fare, dtype: float64


In [ ]:
# Fare = 0 is likely a data error — replacing with median of same Pclass
# because 1st/2nd/3rd class have very different fare ranges as ticket price (fare) depend on pclass,
# a global median would be misleading
df['Fare'] = df.groupby('Pclass')['Fare'].transform(
    lambda x: x.replace(0, x.median())
)

In [35]:
df["Fare"].describe()

count    891.000000
mean      32.674620
std       49.608084
min        4.012500
25%        7.925000
50%       14.500000
75%       31.275000
max      512.329200
Name: Fare, dtype: float64

In [36]:

### Drop Useless Columns droping useless columns
df.drop(columns=["PassengerId", "Name", "Ticket"], inplace=True)

In [38]:
df.columns

Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked', 'HasCabin'],
      dtype='object')

In [39]:
# final checking
df.isnull().sum()

Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
HasCabin    0
dtype: int64

In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    object 
 3   Age       891 non-null    float64
 4   SibSp     891 non-null    int64  
 5   Parch     891 non-null    int64  
 6   Fare      891 non-null    float64
 7   Embarked  891 non-null    object 
 8   HasCabin  891 non-null    int64  
dtypes: float64(2), int64(5), object(2)
memory usage: 62.8+ KB


In [42]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,HasCabin
0,0,3,male,22.0,1,0,7.2500,S,0
1,1,1,female,38.0,1,0,71.2833,C,1
2,1,3,female,26.0,0,0,7.9250,S,0
3,1,1,female,35.0,1,0,53.1000,S,1
4,0,3,male,35.0,0,0,8.0500,S,0


In [43]:
df.to_csv("Titanic-Dataset-Cleaned.csv", index=False)